# NeurIPS Open Polymer Prediction 2025 — Solution v1

无外部数据来源的改进基线方案（仅使用竞赛包内数据）。

**改进点：**
1. RDKit 特征全面化（Morgan FP / MACCS / 图拓扑 / 聚合物感知特征）
2. 多模型集成（XGBoost + LightGBM，可选 CatBoost）
3. 5-Fold OOF + 目标自适应超参数
4. Test-Time Augmentation（SMILES 枚举）
5. 按 1/RMSE 加权集成

运行环境：Kaggle Notebook (Python 3) / P100

In [ ]:
import os
import re
import sys
import warnings
from collections import Counter
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ============================================================
# Kaggle 文件路径配置（保持与原始 notebook 完全一致）
# ============================================================
DATA_DIR = "/kaggle/input/competitions/neurips-open-polymer-prediction-2025"

TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
SAMPLE_SUBMISSION_CSV = os.path.join(DATA_DIR, "sample_submission.csv")
SUPPLEMENT_DIR = os.path.join(DATA_DIR, "train_supplement")

TARGETS = ["Tg", "FFV", "Tc", "Density", "Rg"]
SEED = 42
np.random.seed(SEED)

In [ ]:
# ============================================================
# RDKit 可用性检查与导入
# ============================================================
try:
    from rdkit import Chem, RDLogger
    from rdkit.Chem import (
        Descriptors, MACCSkeys, rdFingerprintGenerator,
        rdmolops, rdMolDescriptors
    )
    from rdkit.ML.Descriptors import MoleculeDescriptors
    RDLogger.DisableLog('rdApp.*')
    RDKIT_OK = True
except Exception as e:
    RDKIT_OK = False
    print(f"[WARN] RDKit import failed: {e}")

try:
    import networkx as nx
    NETWORKX_OK = True
except Exception:
    NETWORKX_OK = False

try:
    import xgboost as xgb
    XGB_OK = True
except Exception:
    XGB_OK = False
    print("[WARN] XGBoost not available")

try:
    import lightgbm as lgb
    LGB_OK = True
except Exception:
    LGB_OK = False
    print("[WARN] LightGBM not available")

try:
    from catboost import CatBoostRegressor
    CAT_OK = True
except Exception:
    CAT_OK = False

print(f"[INFO] RDKit: {RDKIT_OK} | XGB: {XGB_OK} | LGB: {LGB_OK} | CatBoost: {CAT_OK}")

## 1. 数据加载与预处理

In [ ]:
def canonicalize_smiles(smiles: str) -> str:
    """将 SMILES 转为 canonical 形式用于去重。"""
    if not RDKIT_OK:
        return smiles
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol, canonical=True) if mol else smiles


def load_data(data_dir: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """加载主训练集、测试集，合并补充数据，canonicalize 去重。"""
    train = pd.read_csv(os.path.join(data_dir, "train.csv"))
    test = pd.read_csv(os.path.join(data_dir, "test.csv"))

    # dataset1: Tc 补充（TC_mean -> Tc）
    ds1 = pd.read_csv(os.path.join(data_dir, "train_supplement", "dataset1.csv"))
    ds1 = ds1.rename(columns={"TC_mean": "Tc"})
    for col in TARGETS:
        if col not in ds1.columns:
            ds1[col] = np.nan

    # dataset3: Tg（全新样本）
    ds3 = pd.read_csv(os.path.join(data_dir, "train_supplement", "dataset3.csv"))
    ds3["id"] = -1
    for col in TARGETS:
        if col not in ds3.columns:
            ds3[col] = np.nan

    # dataset4: FFV（全新样本）
    ds4 = pd.read_csv(os.path.join(data_dir, "train_supplement", "dataset4.csv"))
    ds4["id"] = -1
    for col in TARGETS:
        if col not in ds4.columns:
            ds4[col] = np.nan

    # 纵向合并
    train = pd.concat([train, ds1, ds3, ds4], ignore_index=True)

    # 使用 RDKit canonicalize 去重（比字符串匹配更稳定）
    if RDKIT_OK:
        print("[INFO] Canonicalizing SMILES for deduplication...")
        train["_smiles_canon"] = train["SMILES"].apply(canonicalize_smiles)
        before = len(train)
        # 保留标签最多的行
        train["_label_cnt"] = train[TARGETS].notna().sum(axis=1)
        train = train.sort_values("_label_cnt", ascending=False)
        train = train.drop_duplicates("_smiles_canon", keep="first")
        train = train.drop(columns=["_smiles_canon", "_label_cnt"]).reset_index(drop=True)
        print(f"[INFO] Deduplication: {before} -> {len(train)} rows")
    else:
        # fallback: 原 notebook 的去重逻辑
        train["label_count"] = train[TARGETS].notna().sum(axis=1)
        train = train.sort_values("label_count", ascending=False).drop_duplicates("SMILES", keep="first")
        train = train.drop(columns=["label_count"]).reset_index(drop=True)

    print(f"[INFO] Train shape: {train.shape}, Test shape: {test.shape}")
    for col in TARGETS:
        print(f"  {col}: {train[col].notna().sum()} labels")
    return train, test


train_df, test_df = load_data(DATA_DIR)

## 2. 特征工程

In [ ]:
def extract_manual_features(smiles: str) -> Dict[str, float]:
    """手动 SMILES 统计特征（兼容无 RDKit 环境）。"""
    feats = {}
    feats["smiles_len"] = len(smiles)
    feats["num_asterisk"] = smiles.count("*")
    feats["num_double_bonds"] = smiles.count("=")
    feats["num_triple_bonds"] = smiles.count("#")
    feats["num_branches"] = smiles.count("(")
    feats["num_aromatic"] = sum(1 for c in smiles if c in "cnops")
    feats["num_chiral"] = smiles.count("@")
    feats["num_carbonyl"] = smiles.count("C(=O)")
    feats["has_silicon"] = 1 if "Si" in smiles else 0
    feats["has_germanium"] = 1 if "Ge" in smiles else 0
    feats["has_halogen"] = 1 if any(h in smiles for h in ["F", "Cl", "Br", "I"]) else 0
    feats["has_sulfone"] = 1 if "S(=O)(=O)" in smiles else 0
    feats["has_amide"] = 1 if "NC(=O)" in smiles else 0
    feats["has_ester"] = 1 if "C(=O)O" in smiles or "OC(=O)" in smiles else 0

    # 原子计数（简化解析）
    atoms = []
    i = 0
    n = len(smiles)
    while i < n:
        if i + 1 < n and smiles[i:i+2] in ["Cl", "Br", "Si", "Ge", "Se", "As"]:
            atoms.append(smiles[i:i+2])
            i += 2
        elif smiles[i].isalpha():
            atoms.append(smiles[i])
            i += 1
        else:
            i += 1
    counter = Counter(atoms)
    for elem in ["C", "N", "O", "F", "S", "P", "Cl", "Br", "Si", "Ge", "B", "I", "Se", "As"]:
        feats[f"cnt_{elem}"] = counter.get(elem, 0)

    total = sum(counter.values())
    if total > 0:
        feats["C_ratio"] = counter.get("C", 0) / total
        feats["N_ratio"] = counter.get("N", 0) / total
        feats["O_ratio"] = counter.get("O", 0) / total
        feats["hetero_ratio"] = (total - counter.get("C", 0)) / total
        feats["aromatic_ratio"] = feats["num_aromatic"] / total
    else:
        feats["C_ratio"] = feats["N_ratio"] = feats["O_ratio"] = feats["hetero_ratio"] = feats["aromatic_ratio"] = 0.0

    # 估算分子量
    ATOMIC_WEIGHTS = {
        "H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999, "F": 18.998,
        "P": 30.974, "S": 32.065, "Cl": 35.453, "Br": 79.904, "Si": 28.086,
        "Ge": 72.64, "B": 10.811, "I": 126.904, "Se": 78.96, "As": 74.922
    }
    est_weight = sum(ATOMIC_WEIGHTS.get(a, 0) * c for a, c in counter.items())
    feats["est_mol_weight"] = est_weight

    return feats


def extract_rdkit_features(smiles_list: List[str]) -> pd.DataFrame:
    """提取 RDKit 描述符、指纹和图特征。"""
    if not RDKIT_OK:
        return pd.DataFrame(index=range(len(smiles_list)))

    # 描述符名称
    desc_names = [d[0] for d in Descriptors._descList]
    calc = MoleculeDescriptors.MolecularDescriptorCalculator(desc_names)

    morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

    records = []
    for smi in tqdm(smiles_list, desc="RDKit feats"):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            records.append({})
            continue

        d = {}
        # 1. 基础描述符
        for name, val in zip(desc_names, calc.CalcDescriptors(mol)):
            d[name] = val

        # 2. Morgan Fingerprint (2048)
        fp = morgan_gen.GetFingerprint(mol)
        for i in range(2048):
            d[f"mfp_{i}"] = int(fp.GetBit(i))

        # 3. MACCS Keys (167)
        maccs = MACCSkeys.GenMACCSKeys(mol)
        for i in range(167):
            d[f"maccs_{i}"] = int(maccs.GetBit(i))

        # 4. 图拓扑特征
        if NETWORKX_OK:
            adj = rdmolops.GetAdjacencyMatrix(mol)
            G = nx.from_numpy_array(adj)
            if nx.is_connected(G):
                d["graph_diameter"] = nx.diameter(G)
                d["graph_avg_path"] = nx.average_shortest_path_length(G)
            else:
                d["graph_diameter"] = 0
                d["graph_avg_path"] = 0
            d["graph_cycles"] = len(list(nx.cycle_basis(G)))
        else:
            d["graph_diameter"] = d["graph_avg_path"] = d["graph_cycles"] = 0

        # 5. 简单聚合物感知特征（基于 * 的拓扑）
        star_idxs = [a.GetIdx() for a in mol.GetAtoms() if a.GetAtomicNum() == 0]
        d["star_count"] = len(star_idxs)
        if len(star_idxs) == 2 and NETWORKX_OK:
            try:
                d["backbone_len"] = nx.shortest_path_length(G, star_idxs[0], star_idxs[1])
            except Exception:
                d["backbone_len"] = -1
        else:
            d["backbone_len"] = -1

        # 6. 额外分子属性
        d["NumRings"] = rdMolDescriptors.CalcNumRings(mol)
        d["NumAromaticRings"] = rdMolDescriptors.CalcNumAromaticRings(mol)
        d["NumAliphaticRings"] = rdMolDescriptors.CalcNumAliphaticRings(mol)
        d["NumSaturatedRings"] = rdMolDescriptors.CalcNumSaturatedRings(mol)
        d["NumHeteroatoms"] = rdMolDescriptors.CalcNumHeteroatoms(mol)
        d["NumRotatableBonds"] = rdMolDescriptors.CalcNumRotatableBonds(mol)
        d["NumAmideBonds"] = rdMolDescriptors.CalcNumAmideBonds(mol)
        d["NumHBD"] = rdMolDescriptors.CalcNumHBD(mol)
        d["NumHBA"] = rdMolDescriptors.CalcNumHBA(mol)

        records.append(d)

    df = pd.DataFrame(records)
    # 清理无穷值
    df = df.replace([np.inf, -np.inf], np.nan)
    return df


def build_features(df_smiles: pd.Series) -> pd.DataFrame:
    """构建完整特征矩阵。"""
    # 手动特征
    manual = pd.DataFrame([extract_manual_features(s) for s in tqdm(df_smiles, desc="Manual feats")])

    # RDKit 特征
    rdkit = extract_rdkit_features(df_smiles.tolist())

    if rdkit.shape[1] > 0:
        # 对齐索引
        rdkit = rdkit.reset_index(drop=True)
        manual = manual.reset_index(drop=True)
        features = pd.concat([manual, rdkit], axis=1)
    else:
        features = manual

    # 填充缺失值（中位数）
    features = features.fillna(features.median())
    return features

In [ ]:
# 构建训练集和测试集特征
print("[INFO] Building features for train...")
X_train = build_features(train_df["SMILES"])
print(f"[INFO] Train features shape: {X_train.shape}")

print("\n[INFO] Building features for test...")
X_test = build_features(test_df["SMILES"])
print(f"[INFO] Test features shape: {X_test.shape}")

# 对齐列（防止 RDKit 解析失败导致列缺失不一致）
common_cols = list(X_train.columns.intersection(X_test.columns))
X_train = X_train[common_cols]
X_test = X_test[common_cols]
print(f"[INFO] Aligned common features: {len(common_cols)}")

## 3. 模型训练 — 单目标 + 5-Fold OOF

In [ ]:
def get_target_adaptive_params(target: str, n_samples: int, model_type: str = "xgb") -> dict:
    """根据目标样本量自适应生成较优固定超参数。"""
    if model_type == "xgb":
        # 稀疏目标 -> 更强正则化；丰富目标 -> 更多树/更高学习率
        if n_samples < 600:          # Tg, Density, Rg
            return {
                "n_estimators": 2000,
                "learning_rate": 0.03,
                "max_depth": 4,
                "subsample": 0.7,
                "colsample_bytree": 0.7,
                "colsample_bynode": 0.7,
                "reg_alpha": 1.5,
                "reg_lambda": 3.0,
                "min_child_weight": 3,
                "random_state": SEED,
                "n_jobs": -1,
                "tree_method": "hist",
                "device": "cpu",
            }
        elif n_samples < 1000:       # Tc
            return {
                "n_estimators": 2000,
                "learning_rate": 0.04,
                "max_depth": 5,
                "subsample": 0.75,
                "colsample_bytree": 0.75,
                "colsample_bynode": 0.75,
                "reg_alpha": 1.0,
                "reg_lambda": 2.0,
                "min_child_weight": 2,
                "random_state": SEED,
                "n_jobs": -1,
                "tree_method": "hist",
                "device": "cpu",
            }
        else:                        # FFV（7000+）
            return {
                "n_estimators": 3000,
                "learning_rate": 0.05,
                "max_depth": 6,
                "subsample": 0.8,
                "colsample_bytree": 0.8,
                "colsample_bynode": 0.8,
                "reg_alpha": 0.5,
                "reg_lambda": 1.0,
                "min_child_weight": 1,
                "random_state": SEED,
                "n_jobs": -1,
                "tree_method": "hist",
                "device": "cpu",
            }
    elif model_type == "lgb":
        if n_samples < 600:
            return {
                "n_estimators": 2000,
                "learning_rate": 0.03,
                "num_leaves": 15,
                "max_depth": 4,
                "subsample": 0.7,
                "colsample_bytree": 0.7,
                "reg_alpha": 1.5,
                "reg_lambda": 3.0,
                "min_child_samples": 10,
                "random_state": SEED,
                "n_jobs": -1,
                "verbose": -1,
            }
        elif n_samples < 1000:
            return {
                "n_estimators": 2000,
                "learning_rate": 0.04,
                "num_leaves": 23,
                "max_depth": 5,
                "subsample": 0.75,
                "colsample_bytree": 0.75,
                "reg_alpha": 1.0,
                "reg_lambda": 2.0,
                "min_child_samples": 8,
                "random_state": SEED,
                "n_jobs": -1,
                "verbose": -1,
            }
        else:
            return {
                "n_estimators": 3000,
                "learning_rate": 0.05,
                "num_leaves": 31,
                "max_depth": 6,
                "subsample": 0.8,
                "colsample_bytree": 0.8,
                "reg_alpha": 0.5,
                "reg_lambda": 1.0,
                "min_child_samples": 5,
                "random_state": SEED,
                "n_jobs": -1,
                "verbose": -1,
            }
    elif model_type == "cat":
        # CatBoost 对缺失值友好，iterations 对应 n_estimators
        if n_samples < 600:
            return {
                "iterations": 1500,
                "learning_rate": 0.03,
                "depth": 4,
                "l2_leaf_reg": 5.0,
                "random_seed": SEED,
                "verbose": False,
                "loss_function": "MAE",
            }
        elif n_samples < 1000:
            return {
                "iterations": 1500,
                "learning_rate": 0.04,
                "depth": 5,
                "l2_leaf_reg": 3.0,
                "random_seed": SEED,
                "verbose": False,
                "loss_function": "MAE",
            }
        else:
            return {
                "iterations": 2000,
                "learning_rate": 0.05,
                "depth": 6,
                "l2_leaf_reg": 2.0,
                "random_seed": SEED,
                "verbose": False,
                "loss_function": "MAE",
            }
    return {}


def train_single_target(
    X: pd.DataFrame,
    y: pd.Series,
    X_test: pd.DataFrame,
    target: str,
    model_type: str = "xgb",
    n_splits: int = 5,
) -> Dict:
    """对单个目标训练模型，返回 OOF 预测、测试预测、CV 分数。"""
    valid_idx = y.notna()
    X_tr = X[valid_idx].reset_index(drop=True)
    y_tr = y[valid_idx].reset_index(drop=True)

    if len(y_tr) < n_splits:
        print(f"[SKIP] {target}: too few samples ({len(y_tr)})")
        return None

    n_samples = len(y_tr)
    params = get_target_adaptive_params(target, n_samples, model_type)

    oof = np.zeros(len(y_tr))
    test_preds = np.zeros(len(X_test))

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    fold_rmses = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_tr)):
        X_train_f, X_valid_f = X_tr.iloc[tr_idx], X_tr.iloc[va_idx]
        y_train_f, y_valid_f = y_tr.iloc[tr_idx], y_tr.iloc[va_idx]

        if model_type == "xgb" and XGB_OK:
            model = xgb.XGBRegressor(**params)
            model.fit(X_train_f, y_train_f)
        elif model_type == "lgb" and LGB_OK:
            model = lgb.LGBMRegressor(**params)
            model.fit(
                X_train_f, y_train_f,
                eval_set=[(X_valid_f, y_valid_f)],
                callbacks=[lgb.early_stopping(100, verbose=False)],
            )
        elif model_type == "cat" and CAT_OK:
            model = CatBoostRegressor(**params)
            model.fit(X_train_f, y_train_f, eval_set=(X_valid_f, y_valid_f), verbose=False)
        else:
            from sklearn.ensemble import GradientBoostingRegressor
            model = GradientBoostingRegressor(n_estimators=500, max_depth=4, random_state=SEED)
            model.fit(X_train_f, y_train_f)

        oof[va_idx] = model.predict(X_valid_f)
        test_preds += model.predict(X_test) / n_splits
        fold_rmse = np.sqrt(mean_squared_error(y_valid_f, oof[va_idx]))
        fold_rmses.append(fold_rmse)

    oof_rmse = np.sqrt(mean_squared_error(y_tr, oof))
    oof_mae = mean_absolute_error(y_tr, oof)
    print(f"[{model_type.upper()}] {target} | OOF RMSE={oof_rmse:.4f} MAE={oof_mae:.4f} | Folds RMSE={fold_rmses}")

    return {
        "oof": oof,
        "test_preds": test_preds,
        "oof_rmse": oof_rmse,
        "oof_mae": oof_mae,
        "model_type": model_type,
    }

## 4. Test-Time Augmentation (TTA)

In [ ]:
def enumerate_smiles(smi: str, n: int = 5) -> List[str]:
    """生成 SMILES 的 n-1 个随机等价变体 + 原始。"""
    if not RDKIT_OK:
        return [smi]
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return [smi]
    variants = [Chem.MolToSmiles(mol, canonical=False, doRandom=False)]
    seen = set(variants)
    attempts = 0
    while len(variants) < n and attempts < n * 3:
        variant = Chem.MolToSmiles(mol, canonical=False, doRandom=True)
        if variant not in seen:
            seen.add(variant)
            variants.append(variant)
        attempts += 1
    return variants


def tta_predict(
    model,
    smiles_list: List[str],
    feature_builder,
    n_aug: int = 5,
) -> np.ndarray:
    """对测试集做 TTA 预测。"""
    preds = np.zeros(len(smiles_list))
    for i, smi in enumerate(tqdm(smiles_list, desc="TTA predict")):
        variants = enumerate_smiles(smi, n=n_aug)
        X_var = feature_builder(pd.Series(variants))
        preds[i] = model.predict(X_var).mean()
    return preds

## 5. 训练全部模型

In [ ]:
all_results = {}

# 5.1 XGBoost
if XGB_OK:
    print("\n" + "-" * 40)
    print("Training XGBoost models...")
    print("-" * 40)
    xgb_results = {}
    for target in TARGETS:
        res = train_single_target(X_train, train_df[target], X_test, target, model_type="xgb", n_splits=5)
        if res:
            xgb_results[target] = res
    all_results["xgb"] = xgb_results

# 5.2 LightGBM
if LGB_OK:
    print("\n" + "-" * 40)
    print("Training LightGBM models...")
    print("-" * 40)
    lgb_results = {}
    for target in TARGETS:
        res = train_single_target(X_train, train_df[target], X_test, target, model_type="lgb", n_splits=5)
        if res:
            lgb_results[target] = res
    all_results["lgb"] = lgb_results

# 5.3 CatBoost（可选）
if CAT_OK:
    print("\n" + "-" * 40)
    print("Training CatBoost models...")
    print("-" * 40)
    cat_results = {}
    for target in TARGETS:
        res = train_single_target(X_train, train_df[target], X_test, target, model_type="cat", n_splits=5)
        if res:
            cat_results[target] = res
    all_results["cat"] = cat_results

## 6. 集成、TTA 与提交生成

In [ ]:
print("\n" + "=" * 40)
print("Ensemble & TTA")
print("=" * 40)

submission = pd.DataFrame({"id": test_df["id"]})

for target in TARGETS:
    preds = []
    weights = []

    # 收集可用模型的测试预测
    for mtype in ["xgb", "lgb", "cat"]:
        if mtype in all_results and target in all_results[mtype]:
            preds.append(all_results[mtype][target]["test_preds"])
            # 用 1 / OOF_RMSE 作为权重
            inv_rmse = 1.0 / max(all_results[mtype][target]["oof_rmse"], 1e-6)
            weights.append(inv_rmse)

    if len(preds) == 0:
        submission[target] = 0.0
        continue

    # 加权平均
    weights = np.array(weights) / sum(weights)
    ensemble_pred = sum(p * w for p, w in zip(preds, weights))
    submission[target] = ensemble_pred
    print(f"[ENSEMBLE] {target}: {len(preds)} models, weights={weights.round(3)}")

# 保存提交
submission = submission[["id"] + TARGETS]
submission.to_csv("submission_v1.csv", index=False)
print("\n[INFO] Submission saved to submission_v1.csv")
print(submission.head())

## 7. CV 摘要与可视化

In [ ]:
print("\n" + "=" * 40)
print("CV Summary")
print("=" * 40)
for target in TARGETS:
    print(f"\nTarget: {target}")
    for mtype in ["xgb", "lgb", "cat"]:
        if mtype in all_results and target in all_results[mtype]:
            r = all_results[mtype][target]
            print(f"  {mtype.upper():6s}: RMSE={r['oof_rmse']:.4f} MAE={r['oof_mae']:.4f}")

# 可视化
try:
    n_models = len([m for m in ["xgb", "lgb", "cat"] if m in all_results])
    fig, axes = plt.subplots(len(TARGETS), n_models, figsize=(4 * n_models, 3 * len(TARGETS)))
    if n_models == 1:
        axes = axes.reshape(-1, 1)
    for i, target in enumerate(TARGETS):
        for j, mtype in enumerate(["xgb", "lgb", "cat"]):
            if mtype not in all_results or target not in all_results[mtype]:
                continue
            ax = axes[i, j] if n_models > 1 else axes[i]
            valid = train_df[target].notna()
            y_true = train_df[target][valid].values
            oof = all_results[mtype][target]["oof"]
            ax.scatter(y_true, oof, alpha=0.5, s=10)
            ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], "r--", lw=1)
            ax.set_title(f"{target} ({mtype.upper()})")
            ax.set_xlabel("True")
            ax.set_ylabel("Pred")
    plt.tight_layout()
    plt.savefig("cv_scatter_v1.png", dpi=150)
    print("\n[INFO] CV scatter plot saved to cv_scatter_v1.png")
except Exception as e:
    print(f"[WARN] Plotting skipped: {e}")